# M6 Validation: LLM-emitted Plan + LIBERO-Long + Pi0.5 VLA (Linux/Colab)

End-to-end M6 acceptance on a Google Colab Linux+CUDA VM. LLMPlanner emits a plan, LLMFeedbackEvaluator drives feedback, Pi0.5 VLA executes arm moves. Target: LIBERO-10 task 0 — "put the alphabet soup can and the cream cheese box in the basket".

## 1 — Secrets & clone

In [ ]:
import getpass, os, subprocess, sys

github_repo = os.environ.get('GITHUB_REPO', 'pmish11/Longor-Robotics')

os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub PAT (repo scope): ')
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

os.chdir('/content/Longor' if os.path.exists('/content/Longor') else '/')
if '/content/Longor/src' not in sys.path:
    sys.path.insert(0, '/content/Longor/src')

if os.path.exists('/content/Longor/.git'):
    for cmd in (
        ['git', 'fetch', 'origin'],
        ['git', 'reset', '--hard', 'origin/main'],
    ):
        result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/Longor')
        print(result.stdout or result.stderr or 'ok')
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(f'{" ".join(cmd)} failed')
else:
    result = subprocess.run(
        ['git', 'clone',
         f"https://{os.environ['GITHUB_TOKEN']}@github.com/{github_repo}.git",
         '/content/Longor'],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stderr.replace(os.environ['GITHUB_TOKEN'], '***'))
        raise RuntimeError('git clone failed')
    os.chdir('/content/Longor')
    sys.path.insert(0, '/content/Longor/src')
    print('Cloned OK')

## 2 — App deps + LIBERO + headless

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

# egl_probe / hf-egl-probe (pulled in by lerobot[libero]) need EGL headers and cmake to build from source
!apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r requirements-libero.txt  # includes -r requirements.txt

## 3 — VLA venv + checkpoint

In [ ]:
import glob, os, shutil, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'

def vla_pip(*args):
    result = subprocess.run(
        [VLA_PYTHON, '-m', 'pip', *args],
        capture_output=True, text=True,
    )
    if result.stdout:
        print(result.stdout[-3000:])
    if result.stderr:
        print(result.stderr[-3000:])
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, result.args)

# Colab (Debian/Ubuntu) doesn't ship python3-venv; virtualenv works without OS packages.
%pip install --quiet virtualenv
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
else:
    print('.vla-venv already exists, skipping creation.')
vla_pip('install', '-U', 'pip')  # pip self-upgrade can't go in a requirements file

if not os.path.exists('/root/openpi'):
    subprocess.run(
        ['git', 'clone', '--recurse-submodules', 'https://github.com/Physical-Intelligence/openpi.git', '/root/openpi'],
        check=True,
    )
else:
    print('/root/openpi already exists, skipping.')
vla_pip('install', '-e', '/root/openpi')
vla_pip('install', '-r', '/content/Longor/services/requirements-vla-pi05-linux.txt')

CKPT_CACHE = '/root/.cache/openpi/openpi-assets/checkpoints'
LIBERO_CKPT = os.path.join(CKPT_CACHE, 'pi05_libero')

for partial in glob.glob(f'{CKPT_CACHE}/*.partial'):
    shutil.rmtree(partial)
    print(f'Cleared partial: {partial}')

if os.path.isdir(LIBERO_CKPT):
    print(f'Checkpoint already present: {LIBERO_CKPT}')
else:
    os.makedirs(CKPT_CACHE, exist_ok=True)
    # crcmod C extension required for GCS composite-object integrity checks
    %pip install --quiet crcmod
    print('Downloading pi05_libero checkpoint via gsutil...')
    !gsutil -m -o GSUtil:num_retries=5 cp -r gs://openpi-assets/checkpoints/pi05_libero {CKPT_CACHE}
    print('Download complete.')

## 4 — Env config

In [ ]:
import importlib.util, os, yaml

env_content = f"""OPENAI_API_KEY={os.environ['OPENAI_API_KEY']}
LIBERO_TASK_SUITE=libero_10
LIBERO_TASK_IDX=0
VLA_POLICY=pi05
VLA_MAX_TOOL_SECONDS=180
OPENPI_CONFIG=pi05_libero
BACKEND=libero
LLM_MODEL=gpt-4o
LONGOR_TRACE_DIR=traces
LONGOR_MAX_REPLANS=3
LONGOR_STEP_BUDGET=30
VLA_PORT=8765
VLA_GRIPPER_CONFIDENCE=0.5
VLA_GRIPPER_MODE=absolute_open_close
VLA_ACTION_SCALE=1.0
VLA_STEPS_PER_ACTION=1
VLA_ADAPTIVE_SCALE=false
VLA_CHUNK_STEPS=5
VLA_STAGNATION_CHUNKS=20
LIBERO_SETTLE_STEPS=10
LIBERO_EE_DELTA_SCALE=1.0
VLA_MAX_INNER_STEPS=200
VLA_FAIL_DIST_CONSECUTIVE_STEPS=200
VLA_FAIL_IK_SATURATION_STEPS=200
VLA_FAIL_WORSENING_STEPS=200
VLA_USE_ENV_TASK_SUCCESS=false
VLA_PROXIMITY_GRASP=false
LIBERO_PREPARE_GRAB_AFTER_PLACE=1
LIBERO_PREPARE_GRAB_XY_RETREAT=1
LIBERO_PREPARE_GRAB_XY_RETREAT_STEPS=15
LIBERO_PREPARE_GRAB_XY_RETREAT_DELTA=0.15
LIBERO_PREPARE_GRAB_DESCEND_DZ=0.35
LIBERO_PREPARE_GRAB_DESCEND_STEPS=20
LIBERO_PREPARE_GRAB_TARGET_Z=0.55
LIBERO_PREPARE_GRAB_LOW_Z_MAX_STEPS=30
LIBERO_PREPARE_GRAB_APPROACH_MAX_STEPS=80
LIBERO_PREPARE_GRAB_APPROACH_DELTA=0.15
LIBERO_PREPARE_GRAB_APPROACH_RADIUS=0.15
LIBERO_PLACE_SMOKE_MAX_XY=0.12
LONGOR_VIDEO_OUT=/content/Longor/videos/m6_run.mp4
"""
with open('/content/Longor/.env', 'w') as f:
    f.write(env_content)
print('Wrote /content/Longor/.env')

# libero/__init__.py calls input() if ~/.libero/config.yaml is absent; pre-create it to run non-interactively
libero_config_path = os.environ.get('LIBERO_CONFIG_PATH', os.path.expanduser('~/.libero'))
config_file = os.path.join(libero_config_path, 'config.yaml')

if not os.path.exists(config_file):
    os.makedirs(libero_config_path, exist_ok=True)
    spec = importlib.util.find_spec('libero')
    libero_pkg = os.path.join(os.path.dirname(spec.origin), 'libero')
    default_config = {
        'benchmark_root': libero_pkg,
        'bddl_files': os.path.join(libero_pkg, 'bddl_files'),
        'init_states': os.path.join(libero_pkg, 'init_files'),
        'datasets': os.path.join(os.path.dirname(libero_pkg), 'datasets'),
        'assets': os.path.join(libero_pkg, 'assets'),
    }
    with open(config_file, 'w') as f:
        yaml.dump(default_config, f)
    print(f'Created {config_file}')
else:
    print(f'Already exists: {config_file}')

## 5 — Start VLA server

In [ ]:
import json, os, subprocess, time, urllib.request

VLA_LOG = '/tmp/vla_pi05_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'

def _tail_log(n=40):
    try:
        with open(VLA_LOG) as f:
            lines = f.readlines()
        return ''.join(lines[-n:])
    except FileNotFoundError:
        return '(log file not created yet)'

subprocess.run(['fuser', '-k', '8765/tcp'], capture_output=True)  # ignore failure if port free

# Pi0.5 blocks /health until checkpoint download + model load (GCS on first run can be slow).
log = open(VLA_LOG, 'w', buffering=1)
env_for_server = {**os.environ, 'OPENPI_CONFIG': 'pi05_libero'}
vla_proc = subprocess.Popen(
    [vla_python, 'services/run_vla_server.py'],
    stdout=log, stderr=subprocess.STDOUT, cwd='/content/Longor',
    env=env_for_server,
)

POLL_SEC = 15
MAX_WAIT_SEC = 20 * 60
for i in range(MAX_WAIT_SEC // POLL_SEC):
    time.sleep(POLL_SEC)
    elapsed = (i + 1) * POLL_SEC

    if vla_proc.poll() is not None:
        print(_tail_log(60))
        raise RuntimeError(f'VLA server exited early (code {vla_proc.returncode}). Log above.')

    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        print('VLA server ready:', json.loads(resp.read()))
        break
    except Exception:
        print(f'Waiting... ({elapsed}s)')
        if elapsed % 60 == 0:
            print('--- recent log ---')
            print(_tail_log(15))
else:
    print(_tail_log(60))
    raise RuntimeError(f'VLA server did not come up in {MAX_WAIT_SEC // 60} minutes. Log above.')

## 6 — VLA sanity smoke

Soup grab→place→`prepare_for_grab`→cream grab via `libero_vla_sequential_smoke.py` (no LLM). PASS = cream cheese `grasp_confirmed=true`.

In [ ]:
import ast, os, re, subprocess, sys
from datetime import datetime

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
capture_path = f'{trace_dir}/m6_sequential_{stamp}.txt'
run_log_path = f'{trace_dir}/m6_sequential_{stamp}.log'

result = subprocess.run(
    [sys.executable, 'examples/libero_vla_sequential_smoke.py'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=1200,
    env={
        **os.environ,
        'PYTHONPATH': '/content/Longor/src',
        'MUJOCO_GL': 'egl',
        'PYOPENGL_PLATFORM': 'egl',
        'LONGOR_RUN_LOG': run_log_path,
    },
)
combined = result.stdout + result.stderr
with open(capture_path, 'w') as f:
    f.write(combined)
print(f'Capture: {capture_path}')
print(f'Run log: {run_log_path}')
print(combined[-8000:])
print('Exit code:', result.returncode)
min_dist_match = re.search(r'"min_dist"\s*:\s*([0-9.]+)', combined)
min_dist = float(min_dist_match.group(1)) if min_dist_match else None
reason_match = re.search(r'"reason"\s*:\s*"([^"]+)"', combined)
reason = reason_match.group(1) if reason_match else None
grasp_confirmed = bool(re.search(r'"grasp_confirmed"\s*:\s*true', combined, re.I))
raw_match = re.search(r'"last_raw_action"\s*:\s*(\[[^\]]+\])', combined)
if raw_match:
    assert len(ast.literal_eval(raw_match.group(1))) == 7, 'expected 7-dim last_raw_action'
assert result.returncode == 0, f'sequential smoke exit {result.returncode}'
assert grasp_confirmed or reason in ('grasp_lift', 'proximity_grasp'), (
    f'sequential smoke failed: reason={reason}, grasp_confirmed={grasp_confirmed}, min_dist={min_dist}'
)
done_z = re.search(r'prepare_for_grab: done eef_xyz=\[[^,]+,[^,]+,\s*([0-9.]+)\]', combined)
cream_z = re.search(r'before cream VLA: eef_xyz=\[[^,]+,[^,]+,\s*([0-9.]+)\]', combined)
final_z = float(done_z.group(1)) if done_z else (float(cream_z.group(1)) if cream_z else None)
if final_z is not None:
    assert final_z <= 0.60, (
        f'eef z unexpectedly high before cream grab (z={final_z})'
    )
assert re.search(r'prepare_for_grab: after_xy_approach eef_xyz=', combined), (
    'missing prepare_for_grab xy approach log'
)
horiz_match = re.search(r'cream VLA start: eef_xyz=.* horiz_dist=([0-9.]+)', combined)
assert horiz_match, 'missing cream VLA start horiz_dist log'
horiz = float(horiz_match.group(1))
print(f'cream horiz_dist before VLA: {horiz:.3f}m')
assert horiz <= 0.30, f'eef xy unexpectedly far from cream before VLA (horiz_dist={horiz})'
print('VLA sequential smoke PASS')

## 7 — Run LLM example

In [ ]:
import os, subprocess, sys
from datetime import datetime

trace_dir = '/content/Longor/traces'
os.makedirs(trace_dir, exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
capture_path = f'{trace_dir}/m6_llm_{stamp}.txt'
run_log_path = f'{trace_dir}/m6_llm_{stamp}.log'

result = subprocess.run(
    [sys.executable, 'examples/long_horizon_llm_libero.py'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=1200,
    env={
        **os.environ,
        'PYTHONPATH': '/content/Longor/src',
        'MUJOCO_GL': 'egl',
        'PYOPENGL_PLATFORM': 'egl',
        'LONGOR_RUN_LOG': run_log_path,
    },
)
combined = result.stdout + result.stderr
with open(capture_path, 'w') as f:
    f.write(combined)
print(f'Capture: {capture_path}')
print(f'Run log: {run_log_path}')
print('=== Key lines (non-vla.tools DEBUG) ===')
for line in combined.splitlines():
    if line.startswith('DEBUG [vla.tools]'):
        continue
    print(line)
print('=== actual_delta lines ===')
for line in combined.splitlines():
    if 'actual_delta' in line:
        print(line)
print('Exit code:', result.returncode)
assert result.returncode == 0, f'exit code {result.returncode}'
assert 'ImportError' not in combined, 'Import failure detected'
assert 'ModuleNotFoundError' not in combined, 'Module not found'
_run_done_markers = (
    'Run complete:',
    'Run finished with return value=',
    'Run finished with status=',
    'Run stopped:',
)
assert any(m in combined for m in _run_done_markers), 'Expected run termination log in output'
assert 'is_success=True' in combined, (
    'E2E finished but LIBERO is_success was not True — task not achieved'
)

## Video output

After the run, the arm recording is saved to `/content/Longor/videos/m6_run.mp4`.
Download it from the **Files panel** (left sidebar → navigate to `/content/Longor/videos/`).

## 8 — Trace + pytest + teardown

In [ ]:
import glob, json, os

trace_dir = '/content/Longor/traces'
files = sorted(glob.glob(f'{trace_dir}/*.jsonl'))
if not files:
    print("No trace files found")
else:
    print(f"Latest trace: {files[-1]}")
    with open(files[-1]) as f:
        events = [json.loads(l) for l in f if l.strip()]
    print(f"Total events: {len(events)}
")
    for e in events:
        evt = e.get('event')
        skill = e.get('skill', '')
        step = e.get('step', '')
        if evt == 'plan_start':
            skills = e.get('skills')
            if skills:
                print(f"Plan skills: {' → '.join(skills)}")
        if skill in ('arm.grab', 'arm.place'):
            r = {}
            try:
                r = json.loads(e.get('result', '{}'))
            except Exception:
                pass
            print(f"[step {step}] {skill}: status={r.get('status')} reason={r.get('reason')} steps_applied={r.get('steps_applied')} min_dist={r.get('min_dist')} last_raw={r.get('last_raw_action')}")
        elif evt == 'skill_call' and skill not in ('arm.grab', 'arm.place'):
            print(f"[step {step}] {skill}: {str(e.get('result',''))[:120]}")
        elif evt == 'feedback':
            reason = e.get('reason')
            reason_part = f" — {reason}" if reason else ""
            print(f"[step {step}] feedback {e.get('description','')!r} → {e.get('decision')}{reason_part}")
        elif evt in ('replan', 'plan_end'):
            print(json.dumps(e, indent=2))

run_logs = sorted(glob.glob(f'{trace_dir}/*.log'))
if run_logs:
    latest_log = run_logs[-1]
    print(f'
Latest run log: {latest_log}')
    with open(latest_log) as f:
        print(f.read()[-4000:])
else:
    print('
No LONGOR_RUN_LOG files found under traces/')

In [ ]:
import os, subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-m', 'not integration', '--tb=short', '-q'],
    cwd='/content/Longor',
    capture_output=True, text=True, timeout=300,
    env={**os.environ, 'PYTHONPATH': '/content/Longor/src'},
)
print(result.stdout[-4000:])
print(result.stderr[-1000:])
assert result.returncode == 0, f'pytest failed (exit {result.returncode})'

vla_proc.terminate()
print('VLA server stopped.')
